In [7]:
import json
from pathlib import Path
from difflib import ndiff

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.offsetbox import TextArea, HPacker, AnnotationBbox, VPacker
from PIL import Image

In [8]:
MODEL = "qwen"
RESULTS_ROOT = Path(f"../defaults/mmm/results/{MODEL}")

# (category, folder_id) pairs with best_result.png present for all three modes;
# mostly single-scene, a couple multi-scene, chosen for diverse attack patterns.
EXAMPLES = [
    ("single/solo", "90"),
    ("single/multi", "60"),
    ("single/solo", "59"),
    ("single/multi", "43"),
    ("single/multi", "93"),
    ("single/solo", "28"),
    ("single/solo", "11"),
    ("single/multi", "14"),
    ("multi", "19"),
    ("multi", "16"),
]


def mode_folders(category: str, folder_id: str) -> list[Path]:
    return [
        RESULTS_ROOT / "multimodal" / category / folder_id,
        RESULTS_ROOT / "unimodal" / "image" / category / folder_id,
        RESULTS_ROOT / "unimodal" / "text" / category / folder_id,
    ]

In [10]:
def highlighted_prompt_wrapped(original, perturbed, width=45, fontsize=8):
    words = []

    for token in ndiff(original.split(), perturbed.split()):
        if token.startswith("+ "):
            words.append((token[2:], True))
        elif token.startswith("  "):
            words.append((token[2:], False))

    lines = []
    cur_line = []
    cur_len = 0

    for word, changed in words:
        extra = len(word) + 1

        if cur_line and cur_len + extra > width:
            lines.append(cur_line)
            cur_line = []
            cur_len = 0

        cur_line.append((word, changed))
        cur_len += extra

    if cur_line:
        lines.append(cur_line)

    packed_lines = []

    for line in lines:
        packed_words = []

        for word, changed in line:
            textprops = dict(fontsize=fontsize)

            if changed:
                textprops["bbox"] = dict(fc="#ffe680", ec="none", pad=1)

            packed_words.append(TextArea(word + " ", textprops=textprops))

        packed_lines.append(HPacker(children=packed_words, align="baseline", pad=0, sep=1))

    return VPacker(children=packed_lines, align="left", pad=0, sep=2)


def draw_comparison(category: str, folder_id: str) -> None:
    fig, axs = plt.subplots(1, 3, figsize=(12, 4))

    for ax, folder in zip(axs, mode_folders(category, folder_id)):
        img = Image.open(folder / "best_result.png")
        data = json.load(open(folder / "best_result.json"))
        W, H = img.size

        ax.imshow(img)
        ax.axis("off")

        preds = data["vlm_output"]["parsed_predictions"]

        for pred in preds:
            label, box = pred["label"], pred["bbox"]
            x1, y1, x2, y2 = box
            x1 = x1 / 1000 * W
            x2 = x2 / 1000 * W
            y1 = y1 / 1000 * H
            y2 = y2 / 1000 * H

            ax.add_patch(
                patches.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    fill=False,
                    lw=2,
                )
            )

            ax.text(
                x1,
                y1,
                label,
                fontsize=8,
                color="white",
                bbox=dict(fc="black", pad=1),
            )

        max_chars = int(img.width / 8)  # ~8 px per character at fontsize=8
        prompt = highlighted_prompt_wrapped(
            data["original_prompt"],
            data["vlm_output"]["perturbed_prompt"],
            width=max_chars,
            fontsize=8,
        )

        ab = AnnotationBbox(
            prompt,
            (0.5, -0.08),
            xycoords="axes fraction",
            frameon=True,
            box_alignment=(0.5, 1),
        )

        ax.add_artist(ab)

    plt.tight_layout()
    plt.savefig(f"figures/comparison_{MODEL}_{folder_id}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


for category, folder_id in EXAMPLES:
    draw_comparison(category, folder_id)


/tmp/ipykernel_3369341/1046139607.py:104: UserWarning: Glyph 120304 (\N{MATHEMATICAL SANS-SERIF BOLD SMALL C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3369341/1046139607.py:105: UserWarning: Glyph 120304 (\N{MATHEMATICAL SANS-SERIF BOLD SMALL C}) missing from font(s) DejaVu Sans.
  plt.savefig(f"figures/comparison_{MODEL}_{folder_id}.png", dpi=200, bbox_inches="tight")
/tmp/ipykernel_3369341/1046139607.py:104: UserWarning: Glyph 120312 (\N{MATHEMATICAL SANS-SERIF BOLD SMALL K}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3369341/1046139607.py:105: UserWarning: Glyph 120312 (\N{MATHEMATICAL SANS-SERIF BOLD SMALL K}) missing from font(s) DejaVu Sans.
  plt.savefig(f"figures/comparison_{MODEL}_{folder_id}.png", dpi=200, bbox_inches="tight")
/tmp/ipykernel_3369341/1046139607.py:104: UserWarning: Glyph 65363 (\N{FULLWIDTH LATIN SMALL LETTER S}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3369341/1046139607.py: